# Serving a model

A [persisted model](saving-and-loading-models.ipynb) becomes useful when other
systems can query it. We'll expose one behind a `POST /predict` HTTP endpoint
using [`axum`](https://docs.rs/axum), Rust's ergonomic web framework.

```{note}
A real server runs forever (it blocks), which a notebook cell can't do. So here
we **define and compile the full server** — request/response types, the handler,
the router — and exercise the inference logic directly. The last section shows
the one-line `axum::serve` call you'd add in an actual binary.
```

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep axum = { version = "0.7" }
:dep tokio = { version = "1", features = ["full"] }
use axum::{Router, routing::post, Json, http::StatusCode};

// The JSON the endpoint accepts and returns.
#[derive(serde::Deserialize)]
struct Features { values: Vec<f64> }
#[derive(serde::Serialize)]
struct Prediction { value: f64 }

// Inference + validation as a plain sync function — reused by the handler AND
// callable directly for testing. In practice it would use a loaded model.
fn score(values: &[f64]) -> Result<f64, String> {
    if values.len() != 2 {
        return Err(format!("expected 2 features, got {}", values.len()));
    }
    Ok(2.0 * values[0] - 1.0 * values[1] + 0.5)  // = the persisted LinearModel
}
println!("types + scoring function defined");

## The handler and router

The handler deserializes the JSON body, validates it (the **same gate** you'd
apply at training time — see the [ETL chapter](../01c-etl/data-preparation.ipynb)),
and returns either a `200` with the prediction or a `400` with a useful message —
never a panic:

In [ ]:
async fn predict_handler(Json(req): Json<Features>) -> Result<Json<Prediction>, (StatusCode, String)> {
    match score(&req.values) {
        Ok(value) => Ok(Json(Prediction { value })),
        Err(msg)  => Err((StatusCode::BAD_REQUEST, msg)),
    }
}

// Wire the handler to a route. This `app` is what a server would serve.
let app: Router = Router::new().route("/predict", post(predict_handler));
println!("router built with POST /predict");

## Exercising the endpoint logic

Calling `score` directly stands in for hitting the endpoint — a valid request
yields a prediction (would be `200`), a malformed one yields a validation error
(would be `400`):

In [ ]:
{
    // Simulate: POST /predict {"values": [3.0, 4.0]}
    match score(&[3.0, 4.0]) {
        Ok(v)  => println!("200 OK  -> prediction = {:.3}", v),
        Err(e) => println!("400 Bad Request -> {}", e),
    }
    // Simulate a malformed request: POST /predict {"values": [3.0]}
    match score(&[3.0]) {
        Ok(v)  => println!("200 OK  -> prediction = {:.3}", v),
        Err(e) => println!("400 Bad Request -> {}", e),
    }
}

## Running it for real

In a binary (not a notebook), you'd bind a port and serve the `app` — two lines:

```rust
#[tokio::main]
async fn main() {
    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000").await.unwrap();
    axum::serve(listener, app).await.unwrap();   // blocks, serving requests
}
```

Then `curl -X POST localhost:3000/predict -d '{"values":[3.0,4.0]}'` returns the
prediction as JSON.

```{note}
This is a **minimal, educational** setup. Production concerns — containerization,
TLS, authentication, scaling, load balancing — are out of scope here; reach for
general Rust web-service resources (the `axum`/`tower` ecosystem) for those.
```

Next: [Monitoring](../09-monitoring/monitoring-a-served-model.ipynb) — watching a
served model for errors, latency, and data drift.